In [0]:
# STEP 1: EXTRACT - Load raw data (it's actually JSON, not CSV)
df = spark.read.json("/databricks-datasets/retail-org/sales_orders/")

df.printSchema()
df.show(5, truncate=False)

root
 |-- clicked_items: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: string (containsNull = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- number_of_line_items: string (nullable = true)
 |-- order_datetime: string (nullable = true)
 |-- order_number: long (nullable = true)
 |-- ordered_products: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- curr: string (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- price: long (nullable = true)
 |    |    |-- promotion_info: struct (nullable = true)
 |    |    |    |-- promo_disc: double (nullable = true)
 |    |    |    |-- promo_id: long (nullable = true)
 |    |    |    |-- promo_item: string (nullable = true)
 |    |    |    |-- promo_qty: long (nullable = true)
 |    |    |-- qty: long (nullable = true)
 |    |    |-- unit: string

In [0]:
from pyspark.sql.functions import col, explode, from_unixtime, round as spark_round, expr

# --- Clean top-level data ---
df_clean = df.dropDuplicates(["order_number"])
df_clean = df_clean.na.drop(subset=["customer_id", "order_number"])

# Safely cast order_datetime to long first (bad/empty values become NULL instead of crashing)
df_clean = df_clean.withColumn("order_datetime_clean", expr("try_cast(order_datetime as bigint)"))
df_clean = df_clean.withColumn("order_date", from_unixtime(col("order_datetime_clean")).cast("timestamp"))

# Drop rows where the date couldn't be parsed
df_clean = df_clean.na.drop(subset=["order_date"])

# --- Flatten the nested ordered_products array ---
df_flat = df_clean.withColumn("product", explode(col("ordered_products"))) \
    .select(
        "order_number",
        "customer_id",
        "customer_name",
        "order_date",
        col("product.id").alias("product_id"),
        col("product.name").alias("product_name"),
        col("product.price").alias("unit_price"),
        col("product.qty").alias("quantity")
    )

# --- Add derived column: total revenue per line item ---
df_flat = df_flat.withColumn("total_revenue", spark_round(col("unit_price") * col("quantity"), 2))

print("Original orders:", df.count())
print("Orders after cleaning bad dates:", df_clean.count())
print("Flattened line items:", df_flat.count())
df_flat.show(10, truncate=False)

Original orders: 4074
Orders after cleaning bad dates: 3955
Flattened line items: 7833
+------------+-----------+------------------------------------------+-------------------+--------------------+---------------------------------------------------------------------------------------------------------------+----------+--------+-------------+
|order_number|customer_id|customer_name                             |order_date         |product_id          |product_name                                                                                                   |unit_price|quantity|total_revenue|
+------------+-----------+------------------------------------------+-------------------+--------------------+---------------------------------------------------------------------------------------------------------------+----------+--------+-------------+
|317568020   |7159905    |TURNER ALSTON,  DENISE                    |2019-08-01 05:36:14|AVpe7vER1cnluZ0-aJu7|Mogitech Keys-To-Go Ultra-Portab

In [0]:
from pyspark.sql.functions import sum as spark_sum, countDistinct, avg

# Revenue by product - top sellers
revenue_by_product = df_flat.groupBy("product_name") \
    .agg(
        spark_sum("total_revenue").alias("total_revenue"),
        spark_sum("quantity").alias("total_units_sold"),
        countDistinct("order_number").alias("num_orders")
    ) \
    .orderBy(col("total_revenue").desc())

revenue_by_product.show(10, truncate=False)

+-------------------------------------------------------------------------------------------+-------------+----------------+----------+
|product_name                                                                               |total_revenue|total_units_sold|num_orders|
+-------------------------------------------------------------------------------------------+-------------+----------------+----------+
|15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)                                    |1191560      |970             |321       |
|YSP-4300 Digital Sound Projector Wireless Active Subwoofer (Black)                         |868287       |914             |283       |
|Zamaha - AVENTAGE 7.2-Ch. 4K Ultra HD A/V Home Theater Receiver - Black                    |489677       |994             |308       |
|Zamaha - MusicCast 5.1-Ch. 4K Ultra HD A/V Home Theater Receiver - Black                   |488784       |996             |290       |
|Cyber-shot DSC-WX220 Digital Camera (Black)    

In [0]:
# Save the cleaned, flattened line-item data
df_flat.write.format("delta").mode("overwrite").saveAsTable("sales_line_items_clean")

# Save the aggregated summary table
revenue_by_product.write.format("delta").mode("overwrite").saveAsTable("revenue_by_product_summary")

print("Tables created successfully")

Tables created successfully


In [0]:
%sql
SELECT * FROM revenue_by_product_summary ORDER BY total_revenue DESC LIMIT 10

product_name,total_revenue,total_units_sold,num_orders
"15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)",1191560,970,321
YSP-4300 Digital Sound Projector Wireless Active Subwoofer (Black),868287,914,283
Zamaha - AVENTAGE 7.2-Ch. 4K Ultra HD A/V Home Theater Receiver - Black,489677,994,308
Zamaha - MusicCast 5.1-Ch. 4K Ultra HD A/V Home Theater Receiver - Black,488784,996,290
Cyber-shot DSC-WX220 Digital Camera (Black),385629,1004,308
Details About Mogitech G920 Xbox Driving Force Racing Wheel For Xbox One And Pc (941000121),345261,954,284
Rony LBT-GPX555 Mini-System with Bluetooth and NFC,343476,890,279
Ramsung J3 - Verizon Prepaid,335266,907,286
CD-C600 5-Disc CD Changer,314055,965,280
h.ear go Wireless Speaker (Viridian Blue),310921,911,291
